# Unigram tokenizer — step by step

 This notebook trains only on a small example and does not write a model.

## 1. Count marked words

`▁` records the beginning of a whitespace-separated word. Repeated words carry larger frequency.

In [8]:
from collections import Counter
from math import log
from pathlib import Path
import sys

project_root=next((path for path in [Path.cwd(),*Path.cwd().parents] if (path/'Unigram'/'code').is_dir()),None)
if project_root is None: raise RuntimeError('Open this notebook inside tokenizer_implementations or Unigram')
if str(project_root) not in sys.path: sys.path.insert(0,str(project_root))

corpus=['hug '*10, 'pug '*5, 'pun '*12, 'bun '*4, 'hugs '*5]
MARKER='▁'
words=Counter(MARKER+word for text in corpus for word in text.split())
print(words)
print('Kernel:',sys.executable)

Counter({'▁pun': 12, '▁hug': 10, '▁pug': 5, '▁hugs': 5, '▁bun': 4})
Kernel: d:\project\.venv\Scripts\python.exe


## 2. Build a large initial vocabulary

Keep every character, then add frequent substrings. Character pieces are protected so every training word remains segmentable.

In [9]:
characters=Counter(); substrings=Counter(); MAX_LENGTH=8
for word,frequency in words.items():
    for start,char in enumerate(word):
        characters[char]+=frequency
        for end in range(start+2,min(len(word),start+MAX_LENGTH)+1):
            substrings[word[start:end]]+=frequency

ranked=sorted(substrings.items(),key=lambda item:(-item[1],item[0]))
frequencies=dict(characters)

for piece,frequency in ranked[:60-len(characters)]: frequencies[piece]=frequency
required=set(characters)
total=sum(frequencies.values())
costs={piece:-log(frequency/total) for piece,frequency in frequencies.items()}
print('Initial pieces:',len(costs)); print(sorted(costs,key=costs.get)[:10])

Initial pieces: 31
['▁', 'u', 'g', 'ug', 'p', 'pu', '▁p', '▁pu', 'n', 'un']


## 3. Viterbi segmentation

At each position, keep the cheapest path found so far. Costs are negative log probabilities, so addition replaces multiplication.

In [10]:
from math import inf
def viterbi(text,model):
    best=[inf]*(len(text)+1); previous=[None]*(len(text)+1); best[0]=0
    max_length=max(map(len,model),default=0)
    for start in range(len(text)):
        for end in range(start+1,min(len(text),start+max_length)+1):
            piece=text[start:end]
            if piece in model and best[start]+model[piece] < best[end]:
                best[end]=best[start]+model[piece]; previous[end]=(start,piece)
    if previous[-1] is None:return None,inf
    pieces=[]; position=len(text)
    while position:
        start,piece=previous[position];pieces.append(piece);position=start
    return pieces[::-1],best[-1]

print(viterbi('▁unhug',{'▁':2,'u':2,'n':2,'un':1,'h':2,'hug':1}))

(['▁', 'un', 'hug'], 4)


## 4. Corpus loss

The corpus loss is the Viterbi loss for each unique word multiplied by its frequency.

In [11]:
def corpus_loss(model):
    return sum(frequency*viterbi(word,model)[1] for word,frequency in words.items())

baseline=corpus_loss(costs)
print('Baseline loss:',baseline)

Baseline loss: 135.77782165706157


## 5. Score token removal

Remove one non-character piece temporarily. A small loss increase means the piece is less useful and can be pruned.

In [12]:
removal=[]
for piece in costs:
    if piece in required:continue
    reduced=dict(costs);del reduced[piece]
    removal.append((corpus_loss(reduced)-baseline,piece))
removal.sort(key=lambda item:(item[0],item[1]))
print('First pieces to prune:',removal[:10])

First pieces to prune: [(0.0, 'bu'), (0.0, 'bun'), (0.0, 'gs'), (0.0, 'hu'), (0.0, 'hug'), (0.0, 'hugs'), (0.0, 'pu'), (0.0, 'pug'), (0.0, 'pun'), (0.0, 'ug')]


## 6. Prune to the target

Remove several low-impact pieces per round and repeat. This direct implementation is deliberately small and slow.

In [13]:
from Unigram.code.training import train_unigram
piece_costs=train_unigram(corpus,target_size=30,initial_size=60,max_piece_length=8,prune_fraction=.2)
print('Learned normal pieces:',len(piece_costs))
print(sorted(piece_costs,key=piece_costs.get)[:15])

Learned normal pieces: 26
['▁', 'u', 'g', 'ug', 'p', 'pu', '▁p', '▁pu', 'n', 'un', 'h', '▁h', '▁hu', '▁hug', 'pun']


## 7. Tokenize, encode, and decode

The project tokenizer adds four special tokens, uses Viterbi for each marked word, and maps the result to IDs.

In [14]:
from Unigram.code import UnigramTokenizer
tokenizer=UnigramTokenizer(piece_costs)
text='hug pug unhug zebra'
tokens=tokenizer.tokenize(text);ids=tokenizer.encode(text)
print('Tokens:',tokens);print('IDs:',ids);print('Decoded:',tokenizer.decode(ids))
assert '<unk>' in tokens

Tokens: ['▁hug', '▁pug', '▁', 'un', 'h', 'ug', '<unk>']
IDs: [17, 25, 5, 13, 14, 7, 0]
Decoded: hug pug unhug <unk>
